<div align="center">

<a href="https://colab.research.google.com/github/utkukose/philosophy-of-ai-NB-lecture/blob/main/weeks/week-03/NB03_computation_and_mind.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Week 03: Computation and Mind: Turing Machines, Symbol Systems and Functionalism

**Philosophy of Artificial Intelligence (11118BLG003)**<br>Prof. Dr. Utku Kose<br>Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye<br>Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)<br>[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

*This course is updated in line with current developments in the field. Last update: September 2026.*

</div>

## How to use this notebook

This notebook is a self-contained version of the week: the lecture text comes first, followed by the hands-on lab. Run the cells in order with Shift+Enter. Exercise cells start with `None` placeholders and print feedback through `check(...)`. Reference solutions sit in collapsed cells: open them only after a genuine attempt. The interactive lab and the self-assessment are available at <https://utkukose.github.io/philosophy-of-ai-NB-lecture/weeks/week-03/lab.html>.

## Overview

The claim that machines could think rests on a view of what thinking is. This week introduces the theory of computation, the physical symbol system hypothesis and functionalism, the doctrines that together make strong AI a coherent possibility [1, 2, 3].

## Learning outcomes

By the end of the week, students are expected to describe a Turing machine and the Church-Turing thesis, to state the physical symbol system hypothesis and the computational theory of mind, to explain functionalism and multiple realisability, and to implement and test simple Turing machines.

## What computation is

In 1936, Turing analysed what a human computer, a person carrying out a calculation by rule, can do, and captured it in an abstract machine [1]. A Turing machine has an unbounded tape divided into cells, a head that reads and writes one symbol at a time and moves left or right, a finite set of states and a transition table that fixes, for each state and symbol, what to write, where to move and which state comes next. Turing also described a universal machine that can simulate any other machine given its description, the idea behind the stored-program computer. Figure 3.1 shows a machine that adds one to a binary number.

In the same year, Church reached equivalent results with a different formalism [4]. The Church-Turing thesis states that every function that can be computed by an effective procedure can be computed by a Turing machine. It is a thesis rather than a theorem, because the notion of an effective procedure is informal, but every alternative formalism proposed since has turned out to be equivalent.

![Figure 3.1](https://raw.githubusercontent.com/utkukose/philosophy-of-ai-NB-lecture/main/weeks/week-03/figures/w03_fig1.png)

*Figure 3.1. A Turing machine for adding one to a binary number: the head moves to the right end, then propagates the carry to the left [1].*

## Symbols, search and the language of thought

Newell and Simon turned computation into a hypothesis about intelligence [2]. Their physical symbol system hypothesis states that a physical symbol system has the necessary and sufficient means for general intelligent action. A physical symbol system is a machine that creates, copies, modifies and destroys symbol structures, and its symbols designate objects and processes. If the hypothesis is true, human intelligence is a form of symbol processing, and a suitably programmed computer can in principle be intelligent.

Fodor developed a related view in philosophy of mind, the computational theory of mind [5]. Thinking consists of computations over mental representations that have a language-like structure, a language of thought. The view explains how thought can be productive, since new thoughts are built from old parts, and it links the meaning of thoughts to their causal roles in reasoning.

## Functionalism and multiple realisability

Putnam compared mental states with the logical states of a Turing machine [3]. A machine state is defined by its relations to inputs, outputs and other states, not by the hardware that realises it. Functionalism generalises the idea: A mental state such as pain is defined by its causal role, what causes it, what it causes and how it interacts with other states. It follows that the same mental state could be realised in very different substrates, such as neurons or silicon. This is multiple realisability, and it is the premise that makes strong AI coherent: If minds are defined by function, the right program on any capable hardware would have a mind.

The lab makes multiple realisability concrete. The same function, adding one to a number, is realised by a Turing machine, by a circuit of logic gates and by a lookup table. The three realisations agree on every input and differ completely in their internal operations.

## When does a physical system compute?

Functionalism faces a puzzle. If computing is merely a matter of mapping physical states onto computational states, almost any physical system could be said to compute almost anything, which would make computationalism trivially true. Piccinini proposed a mechanistic account in response: A physical system computes when it is a mechanism whose function is to process vehicles, such as digits, according to rules that are sensitive to properties of those vehicles [6]. The account separates genuine computers from rocks and walls and gives computationalism empirical content.

The doctrines of this week set up the next one. If thinking is computation and computation is substrate-independent, then running the right program suffices for a mind. Searle's Chinese Room argument, the topic of Week 4, targets exactly this inference.

> **Pause and reflect.** Is a thermostat a physical symbol system? Apply Newell and Simon's definition and Piccinini's criterion and compare the answers.

# Hands-on lab

The notebook implements a Turing machine simulator, verifies that the binary increment machine realises the same function as ordinary arithmetic, runs the two-state busy beaver and asks for two small machines to be analysed [1, 3].

In [ ]:
# Setup: common libraries and the self-check helper used in the exercises.
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
SEED = 2026

def check(name, value, expected, tol=1e-6):
    """Compare an exercise answer with the reference value and print feedback."""
    if value is None:
        print(f"[{name}] Not attempted yet. Replace None with your answer and run the cell again.")
        return
    try:
        ok = bool(np.allclose(np.asarray(value, dtype=float), np.asarray(expected, dtype=float), atol=tol))
    except (TypeError, ValueError):
        ok = value == expected
    print(f"[{name}] " + ("Correct." if ok else "Not yet. Revisit the explanation above and try again."))

## 1. A Turing machine simulator

In [ ]:
def run_tm(table, tape_input, start, halt, blank="_", max_steps=10_000):
    """Run a Turing machine. table maps (state, symbol) to (write, move, next_state)."""
    tape = dict(enumerate(tape_input))
    head, state, steps = 0, start, 0
    while state not in halt and steps < max_steps:
        symbol = tape.get(head, blank)
        write, move, state = table[(state, symbol)]
        tape[head] = write
        head += 1 if move == "R" else -1
        steps += 1
    content = "".join(tape[i] for i in sorted(tape)).strip(blank)
    return content, steps, state

INCREMENT = {("right", "0"): ("0", "R", "right"), ("right", "1"): ("1", "R", "right"), ("right", "_"): ("_", "L", "carry"),
             ("carry", "1"): ("0", "L", "carry"), ("carry", "0"): ("1", "L", "done"), ("carry", "_"): ("1", "L", "done")}
print(run_tm(INCREMENT, "1011", "right", {"done"}))

## 2. Functional equivalence

The machine and Python's integer arithmetic realise the same function if they agree on every input.

In [ ]:
agree = all(int(run_tm(INCREMENT, format(n, "b"), "right", {"done"})[0], 2) == n + 1 for n in range(256))
print("agrees with n + 1 for all n below 256:", agree)
BUSY_BEAVER = {("A", "0"): ("1", "R", "B"), ("A", "1"): ("1", "L", "B"), ("B", "0"): ("1", "L", "A"), ("B", "1"): ("1", "R", "H")}
content, steps, _ = run_tm(BUSY_BEAVER, "", "A", {"H"}, blank="0")
print(f"two-state busy beaver: {content.count('1')} ones written in {steps} steps")

### Your turn, exercise 1

Complete the transition table of a machine that flips every bit of a binary string (0 becomes 1 and 1 becomes 0) and halts in state `done` when it reaches the blank after the input.

In [ ]:
FLIP = {
    # ("go", "0"): ...,
}

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
_FLIP_REFERENCE = {("go", "0"): ("1", "R", "go"), ("go", "1"): ("0", "R", "go"), ("go", "_"): ("_", "L", "done")}

In [ ]:
tests = ["0", "1", "1011", "000111"]
try:
    results = [run_tm(FLIP, t, "go", {"done"})[0] for t in tests]
except KeyError:
    results = None
check("Exercise 1", results, [run_tm(_FLIP_REFERENCE, t, "go", {"done"})[0] for t in tests])

### Your turn, exercise 2

Count the steps that the increment machine needs for inputs of seven ones, `1111111`, which is the worst case for seven digits. Then plot the number of steps against input length for inputs made only of ones and describe the growth.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _steps_reference():
    return run_tm(INCREMENT, "1111111", "right", {"done"})[1]

In [ ]:
steps_worst = None  # your answer
check("Exercise 2", steps_worst, _steps_reference())

## Visual exploration: A Turing machine at work

The cell simulates a four-state, two-symbol machine from the busy beaver literature and draws its space-time diagram: Each row is the tape at one step, and the red dots mark the head. Edit the transition table to explore other machines.

In [ ]:
rules = {("A", 0): (1, 1, "B"), ("A", 1): (1, -1, "B"), ("B", 0): (1, -1, "A"), ("B", 1): (0, -1, "C"),
         ("C", 0): (1, 1, "H"), ("C", 1): (1, -1, "D"), ("D", 0): (1, 1, "D"), ("D", 1): (0, 1, "A")}
tape, head, state, history = {}, 0, "A", []
while state != "H" and len(history) < 500:
    history.append((dict(tape), head))
    write, move, state = rules[(state, tape.get(head, 0))]
    tape[head] = write
    head += move
history.append((dict(tape), head))
cells = [i for t, h in history for i in list(t) + [h]]
lo, hi = min(cells), max(cells)
img = np.array([[t.get(i, 0) for i in range(lo, hi + 1)] for t, _ in history])
plt.figure(figsize=(6, 7)); plt.imshow(img, cmap="Greys", aspect="auto", interpolation="nearest")
plt.plot([h - lo for _, h in history], range(len(history)), "r.", ms=3)
plt.xlabel("tape cell"); plt.ylabel("step"); plt.title("Space-time diagram of a four-state Turing machine"); plt.show()
print(f"halted after {len(history) - 1} steps with {sum(history[-1][0].values())} ones on the tape")

## Self-assessment and reflection

Take the self-assessment in the interactive lab: <https://utkukose.github.io/philosophy-of-ai-NB-lecture/weeks/week-03/lab.html> (tab: Check yourself). Then answer the reflection prompts:

1. If functionalism is true, what would it take for a program to have your beliefs? What would be missing if it had only your input-output behaviour?
2. Which of the three realisations in the lab, if any, seems most like thinking? Why do intuitions differ between them?
3. Is the brain a physical symbol system? Give one consideration for and one against.

## Weekly task and submission

Write about 400 words explaining how the Church-Turing thesis, the physical symbol system hypothesis and functionalism combine to support strong AI. Identify the weakest link in the chain and defend your choice. Attach the notebook with both exercises completed.

The weekly task supports self-learning and builds a personal portfolio. When the course is followed with the instructor during an active semester, the task can be sent together with the exported learning log to utkukose@sdu.edu.tr or utkukose@gmail.com for evaluation.

## Research and report assignment (optional)

**Is the brain a computer?.** Compare the mechanistic account of physical computation with arguments that computation is observer-relative or trivial, and state which view supports functionalism best [3, 6, 7].

This research assignment is optional and supports self-learning. When the related weeks are followed within the course during an active semester, the report can be sent to utkukose@sdu.edu.tr or utkukose@gmail.com for evaluation. Unless the assignment states otherwise, a report has 1500 to 2500 words, follows the structure of an academic paper, cites at least six scholarly or official sources in square brackets and ends with a reference list.

## References

[1] Turing, A. M. (1936). On computable numbers, with an application to the Entscheidungsproblem. *Proceedings of the London Mathematical Society*, s2-42(1), 230-265. <https://doi.org/10.1112/plms/s2-42.1.230>

[2] Newell, A., & Simon, H. A. (1976). Computer science as empirical inquiry: Symbols and search. *Communications of the ACM*, 19(3), 113-126. <https://doi.org/10.1145/360018.360022>

[3] Putnam, H. (1960). Minds and machines. In S. Hook (Eds.), *Dimensions of Mind* (pp. 138-164). New York University Press.

[4] Church, A. (1936). An unsolvable problem of elementary number theory. *American Journal of Mathematics*, 58(2), 345-363. <https://doi.org/10.2307/2371045>

[5] Fodor, J. A. (1975). *The Language of Thought*. Thomas Y. Crowell.

[6] Piccinini, G. (2015). *Physical Computation: A Mechanistic Account*. Oxford University Press.

[7] Searle, J. R. (1990). Is the brain's mind a computer program?. *Scientific American*, 262(1), 26-31. <https://doi.org/10.1038/scientificamerican0190-26>